# Energy Comparison: Interchange vs OMMFFS for Protein Parameterization

Verify that Interchange and OpenMMForceFields produce identical energies when parameterizing proteins via SMIRNOFF template generation (PR #430).

In [1]:
import copy

import numpy as np
import openmm
import openmm.unit
from openff.toolkit import ForceField as OFFForceField
from openff.toolkit import Topology
from openmm.app import ForceField, Modeller, NoCutoff, PDBFile

from openmmforcefields.generators import SMIRNOFFTemplateGenerator
from openmmforcefields.utils import get_data_filename

## Utility Functions

Adapted from `test_template_generators.py`.

In [2]:
ENERGY_UNIT = openmm.unit.kilocalories_per_mole
FORCE_UNIT = openmm.unit.kilocalories_per_mole / openmm.unit.angstroms
ENERGY_DEVIATION_TOLERANCE = 1.0e-2 * ENERGY_UNIT
RELATIVE_FORCE_DEVIATION_TOLERANCE = 1.0e-5


def compute_energy(system, positions):
    """Compute potential energy and force components for an OpenMM system."""
    system = copy.deepcopy(system)
    for index, force in enumerate(system.getForces()):
        force.setForceGroup(index)
    platform = openmm.Platform.getPlatformByName("Reference")
    integrator = openmm.VerletIntegrator(0.001)
    context = openmm.Context(system, integrator, platform)
    context.setPositions(positions)
    context.applyConstraints(integrator.getConstraintTolerance())

    energy = {
        "total": context.getState(getEnergy=True).getPotentialEnergy(),
        "components": {
            system.getForce(i).__class__.__name__: context.getState(
                getEnergy=True, groups=(1 << i)
            ).getPotentialEnergy()
            for i in range(system.getNumForces())
        },
    }
    forces = {
        "total": context.getState(getForces=True).getForces(asNumpy=True),
        "components": {
            system.getForce(i).__class__.__name__: context.getState(
                getForces=True, groups=(1 << i)
            ).getForces(asNumpy=True)
            for i in range(system.getNumForces())
        },
    }
    del context, integrator
    return energy, forces


def get_permutation_indices(system_1, system_2):
    """Compute permutation indices mapping particles between two systems."""
    atoms_1, sites_1, atoms_2, sites_2 = [], [], [], []
    for i in range(system_1.getNumParticles()):
        (sites_1 if system_1.isVirtualSite(i) else atoms_1).append(i)
    for i in range(system_2.getNumParticles()):
        (sites_2 if system_2.isVirtualSite(i) else atoms_2).append(i)

    assert len(atoms_1) == len(atoms_2), f"Atom count mismatch: {len(atoms_1)} vs {len(atoms_2)}"
    assert len(sites_1) == len(sites_2), f"Vsite count mismatch: {len(sites_1)} vs {len(sites_2)}"

    particles_1 = atoms_1 + sites_1
    particles_2 = atoms_2 + sites_2
    perm_12 = [-1] * len(particles_1)
    perm_21 = [-1] * len(particles_1)
    for i1, i2 in zip(particles_1, particles_2):
        perm_12[i1] = i2
        perm_21[i2] = i1
    return perm_12, perm_21


def compare_energies(name, positions, template_system, reference_system):
    """Compare energies and forces between template-generated and reference systems."""
    forces_perm, pos_perm = get_permutation_indices(template_system, reference_system)

    assert template_system.getNumConstraints() == reference_system.getNumConstraints(), (
        f"Constraint count mismatch: {template_system.getNumConstraints()} vs {reference_system.getNumConstraints()}"
    )

    template_energy, template_forces = compute_energy(template_system, positions)
    ref_positions = [positions[i] for i in pos_perm]
    ref_energy, ref_forces = compute_energy(reference_system, ref_positions)

    # Permute reference forces
    ref_forces["total"] = (
        np.array([ref_forces["total"][i].value_in_unit(FORCE_UNIT) for i in forces_perm]) * FORCE_UNIT
    )
    for comp, f in ref_forces["components"].items():
        ref_forces["components"][comp] = (
            np.array([f[i].value_in_unit(FORCE_UNIT) for i in forces_perm]) * FORCE_UNIT
        )

    components = set(ref_energy["components"]) & set(template_energy["components"])

    # Print energy breakdown
    print(f"\n{'Component':<28} {'Template (kcal/mol)':>20} {'Reference (kcal/mol)':>20} {'Delta':>14}")
    print("-" * 84)
    for key in sorted(components):
        t_e = template_energy["components"][key].value_in_unit(ENERGY_UNIT)
        r_e = ref_energy["components"][key].value_in_unit(ENERGY_UNIT)
        print(f"{key:<28} {t_e:20.6f} {r_e:20.6f} {t_e - r_e:14.6e}")
    t_tot = template_energy["total"].value_in_unit(ENERGY_UNIT)
    r_tot = ref_energy["total"].value_in_unit(ENERGY_UNIT)
    delta = template_energy["total"] - ref_energy["total"]
    print("-" * 84)
    print(f"{'TOTAL':<28} {t_tot:20.6f} {r_tot:20.6f} {(t_tot - r_tot):14.6e}")

    # Force comparison
    def norm_sq(x):
        return (x * x).sum() / x.shape[0]

    def relative_deviation(x, y):
        x = x.value_in_unit(FORCE_UNIT)
        y = y.value_in_unit(FORCE_UNIT)
        scale = norm_sq(x) + norm_sq(y)
        return np.sqrt(norm_sq(x - y) / scale) if scale > 0 else 0

    rel_dev = relative_deviation(template_forces["total"], ref_forces["total"])
    print(f"\nRelative force deviation: {rel_dev:.2e}")

    # Assertions
    assert abs(delta) <= ENERGY_DEVIATION_TOLERANCE, (
        f"Energy deviation {delta.in_units_of(ENERGY_UNIT)} exceeds tolerance {ENERGY_DEVIATION_TOLERANCE}"
    )
    assert rel_dev <= RELATIVE_FORCE_DEVIATION_TOLERANCE, (
        f"Force deviation {rel_dev} exceeds tolerance {RELATIVE_FORCE_DEVIATION_TOLERANCE}"
    )
    print(f"PASS: {name}")
    return delta, rel_dev


def propagate_dynamics(positions, system, nsteps=100):
    """Run a few steps of dynamics to generate a perturbed configuration."""
    temperature = 300 * openmm.unit.kelvin
    collision_rate = 1.0 / openmm.unit.picoseconds
    timestep = 1.0 * openmm.unit.femtoseconds
    integrator = openmm.LangevinIntegrator(temperature, collision_rate, timestep)
    platform = openmm.Platform.getPlatformByName("Reference")
    context = openmm.Context(system, integrator, platform)
    context.setPositions(positions)
    openmm.LocalEnergyMinimizer.minimize(context)
    integrator.step(nsteps)
    return context.getState(getPositions=True).getPositions()

## Energy Comparison Loop

In [3]:
ff_file = "openff_no_water-3.0.0-alpha0.offxml"
pdb_files = ["test-ala-3.pdb", "test-aa.pdb"]

for pdb_name in pdb_files:
    print(f"\n{'='*84}")
    print(f"Testing: {pdb_name}")
    print(f"{'='*84}")

    data_file = get_data_filename(pdb_name)
    pdb = PDBFile(data_file)
    off_top = Topology.from_pdb(data_file)

    # --- Interchange (reference) path ---
    openff_ff = OFFForceField(ff_file)
    interchange_system = openff_ff.create_openmm_system(off_top)

    # --- OMMFFS (template generator) path ---
    molecule = off_top.molecule(0)
    generator = SMIRNOFFTemplateGenerator(molecules=[molecule], forcefield=ff_file)
    openmm_ff = ForceField()
    openmm_ff.registerTemplateGenerator(generator.generator)

    modeller = Modeller(pdb.topology, pdb.positions)
    modeller.addExtraParticles(openmm_ff)
    ommffs_system = openmm_ff.createSystem(modeller.topology, nonbondedMethod=NoCutoff)

    # Compare at PDB positions
    print("\n--- At PDB positions ---")
    compare_energies(f"{pdb_name} (PDB positions)", modeller.positions, ommffs_system, interchange_system)

    # Compare after dynamics (perturbed geometry)
    print("\n--- After 100 steps of dynamics ---")
    new_positions = propagate_dynamics(modeller.positions, ommffs_system)
    compare_energies(f"{pdb_name} (perturbed)", new_positions, ommffs_system, interchange_system)


Testing: test-ala-3.pdb


/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


[10:50:47] WARNING: Proton(s) added/removed



/Users/jeffreywagner/micromamba/envs/ommffs-dev/lib/python3.13/site-packages/openff/interchange/smirnoff/_create.py:151: PresetChargesAndVirtualSitesWarning: Preset charges were provided (via `charge_from_molecules`) alongside a force field that includes virtual site parameters. Note that virtual sites will be applied charges from the force field and cannot be given preset charges. Virtual sites may also affect the charges of their orientation atoms, even if those atoms are given preset charges.
  warnings.warn(


[10:50:48] WARNING: Proton(s) added/removed




--- At PDB positions ---

Component                     Template (kcal/mol) Reference (kcal/mol)          Delta
------------------------------------------------------------------------------------
CMMotionRemover                          0.000000             0.000000   0.000000e+00
HarmonicAngleForce                      19.590490            19.590490   0.000000e+00
HarmonicBondForce                        2.480995             2.480995   0.000000e+00
NonbondedForce                          29.707239            29.707239   0.000000e+00
PeriodicTorsionForce                    11.171106            11.171106   0.000000e+00
------------------------------------------------------------------------------------
TOTAL                                   62.949831            62.949831   0.000000e+00

Relative force deviation: 2.43e-16
PASS: test-ala-3.pdb (PDB positions)

--- After 100 steps of dynamics ---

Component                     Template (kcal/mol) Reference (kcal/mol)          Delta
----

[10:50:49] WARNING: Proton(s) added/removed



[10:50:51] WARNING: Proton(s) added/removed




--- At PDB positions ---

Component                     Template (kcal/mol) Reference (kcal/mol)          Delta
------------------------------------------------------------------------------------
CMMotionRemover                          0.000000             0.000000   0.000000e+00
HarmonicAngleForce                     386.251233           386.251233   0.000000e+00
HarmonicBondForce                       76.549213            76.549213   0.000000e+00
NonbondedForce                        -555.560754          -555.560754   0.000000e+00
PeriodicTorsionForce                   231.349602           231.349602   0.000000e+00
------------------------------------------------------------------------------------
TOTAL                                  138.589295           138.589295  -2.842171e-14

Relative force deviation: 1.63e-16
PASS: test-aa.pdb (PDB positions)

--- After 100 steps of dynamics ---



Component                     Template (kcal/mol) Reference (kcal/mol)          Delta
------------------------------------------------------------------------------------
CMMotionRemover                          0.000000             0.000000   0.000000e+00
HarmonicAngleForce                     182.700296           182.700296   0.000000e+00
HarmonicBondForce                       28.689770            28.689770  -7.105427e-15
NonbondedForce                        -714.254521          -714.254521   0.000000e+00
PeriodicTorsionForce                   180.662441           180.662441   0.000000e+00
------------------------------------------------------------------------------------
TOTAL                                 -322.202014          -322.202014   5.684342e-14

Relative force deviation: 2.97e-16
PASS: test-aa.pdb (perturbed)


In [4]:
print("All energy comparisons passed!")

All energy comparisons passed!
